# Удаление космических лучей

Почти все изображения с CCD будут содержать некоторое количество космических лучей, заряженных частиц, которые бомбардируют верхние слои земной атмосферы. Некоторые из них пройдут через атмосферу и попадут в ваш детектор (скорость космических лучей будет намного выше для камер в космосе). Хотя количество космических лучей примерно пропорционально времени экспозиции, космические лучи будут присутствовать даже в bias-кадрах, в которых микросхема считывается немедленно.

Этот ноутбук объясняет, как удалять космические лучи из калибровочных изображений и научных изображений.

## Удаление из калибровочных изображений

Наиболее удобный способ удалить космические лучи из калибровочных изображений (bias, dark и flat-изображений) - это правильно их комбинировать. Космические лучи, по своей природе, являются случайными событиями, которые будут влиять на разные части каждого из калибровочных изображений. Пиксель, затронутый космическим лучом в одном из тёмных изображений, например, почти наверняка *не* будет затронут космическим лучом в каком-либо из других тёмных изображений.

Комбинирование этих изображений путём усреднения (чтобы уменьшить шум как можно больше) и сигма-отсечения (чтобы исключить экстремальные пиксели в отдельных изображениях, таких как один с космическим лучом) удалит космический луч из комбинированного тёмного изображения. Альтернативой было бы комбинирование изображений с использованием медианы. Подробное описание каждого варианта обсуждается в [разделе о комбинировании изображений](01-06-Image-combination.ipynb).

Метод, описанный ниже для удаления космических лучей из научных изображений, не будет хорошо работать для их удаления из калибровочных изображений и является ненужным, потому что они могут быть удалены путём правильного комбинирования изображений.

## Удаление из научных изображений

Одна хорошая техника удаления космических лучей из изображения - это метод [LACosmic](http://www.astro.yale.edu/dokkum/lacosmic/), первоначально разработанный и реализованный для IRAF [Питером Г. ван Доккумом](https://www.pietervandokkum.com/). Оригинальная статья, описывающая метод, который использует острые края космических лучей для их отличия от других источников в изображении, находится [здесь](http://adsabs.harvard.edu/abs/2001PASP..113.1420V).

Конкретная реализация LACosmic, используемая здесь, это астрофизический пакет, связанный с astropy [Astro-SCRAPPY](https://github.com/astropy/astroscrappy). Если вы используете этот код для удаления космических лучей, вы должны цитировать как оригинальную статью, так и [Astro-SCRAPPY](https://github.com/astropy/astroscrappy) (детали цитирования находятся на его веб-сайте). Код ниже никогда не импортирует [Astro-SCRAPPY](https://github.com/astropy/astroscrappy) напрямую, потому что `ccdproc` предоставляет обёртку для неё, поэтому мы обратили на неё внимание здесь.

### Очень важные замечания об использовании LACosmic

Перед использованием техники LACosmic нужно знать о нескольких вещах. Они взяты из рекомендаций, которые van Dokkum предоставляет в [Заметках для пользователей](http://www.astro.yale.edu/dokkum/lacosmic/) и оригинальной статье.

1. Изображения должны быть вычтены с bias и dark.
2. Изображения должны быть плоскоёл fielded, хотя техника может быть применена без flat fielding.
3. Изображения должны **не** иметь вычтённый небо перед обнаружением космических лучей.
4. Уровень шума в изображении должен быть точно измерен.
5. Изображение и шум должны быть в одних и тех же единицах, обычно электронов.
1. Если есть пиксели, которые известны как плохие (например, горячие пиксели, пиксели, идентифицированные `ccdmask`), они должны быть замаскированы перед обнаружением космических лучей.

In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

from astropy.nddata import CCDData
from astropy.nddata import block_replicate
from astropy import units as u
import ccdproc as ccdp
from photutils.segmentation import detect_sources

from convenience_functions import show_image, display_cosmic_rays

In [ ]:
# Use custom style for larger fonts and figures
plt.style.use('guide.mplstyle')

### Входное изображение

Изображение, которое мы будем использовать в этом ноутбуке, - это одно из уменьшенных изображений из примера 2 в предыдущих ноутбуках. Это изображение поля экзопланеты KELT-16 b, снятое с помощью термоэлектрически охлаждаемого CCD с шумом считывания 10$e^-$ и коэффициентом усиления $1,5~e^-$/ADU.

In [ ]:
ex2_path = Path('example2-reduced')

ccd = CCDData.read(ex2_path / 'kelt-16-b-S001-R001-C084-r.fit')

In [ ]:
show_image(ccd, cmap='gray')

Единица этого изображения - ADU, поэтому нам нужно умножить на коэффициент усиления, чтобы преобразовать в электроны.

In [ ]:
ccd = ccdp.gain_correct(ccd, 1.5 * u.electron / u.adu)

### Чтение и применение масок

Две маски были рассчитаны ранее. Горячие пиксели, тёмный ток которых не может быть исправлен, были [идентифицированы путём сравнения тёмных кадров разного времени экспозиции](08-01-Identifying-hot-pixels.ipynb). [Функция `ccdmask` была использована](08-02-Creating-a-mask) для идентификации других плохих пикселей; один пример был столбец пикселей на левой стороне изображения.

Мы читаем обе маски, которые одинаковы для всех изображений, и комбинируем, используя логическое "ИЛИ".

In [ ]:
dark_mask = CCDData.read(ex2_path / 'mask_from_dark_current.fits', unit=u.dimensionless_unscaled)

In [ ]:
ccdmask_mask = CCDData.read(ex2_path / 'mask_from_ccdmask.fits', unit=u.dimensionless_unscaled)

In [ ]:
combined_mask = dark_mask.data | ccdmask_mask.data

In [ ]:
show_image(combined_mask, cmap='gray')

Исключение этих пикселей из обнаружения космических лучей гарантирует, что идентифицируются только космические лучи.

Маска теперь применяется к изображению KELT-16 b.

In [ ]:
ccd.mask = combined_mask

### Запуск LACosmic

Фактический вызов LACosmic довольно удобен. Ключевые параметры - `readnoise`, шум считывания, и `sigclip`, который определяет, насколько выше фона должен быть пиксель, чтобы считать его космическим лучом. Нет твёрдого правила для выбора правильного значения `sigclip`. В оригинальной статье рекомендуется значение 5, но для этого изображения оно обнаруживает несколько тысяч пикселей, загрязнённых космическими лучами. Это неправдоподобно для изображения, снятого камерой в 1000 футах над уровнем моря.

Более высокие значения `sigclip` уменьшают количество найденных космических лучей. Значение, использованное ниже, 7, казалось хорошо работать для этого изображения, обнаруживая всего примерно 70 пикселей, которые являются космическими лучами, и пару десятков кандидатов космических лучей, которые простираются на несколько пикселей.

Функция [`cosmicray_lacosmic`](https://ccdproc.readthedocs.io/en/latest/api/ccdproc.cosmicray_lacosmic.html#ccdproc.cosmicray_lacosmic) из `ccdproc` возвращает новое изображение, в котором маска `True` для пикселей, в которых был обнаружен космический луч, и `False` в противном случае. Данные в новом изображении имеют значения в пикселях, в которых были идентифицированы космические лучи, заменённые путём интерполяции соседних пикселей.

Мы посмотрим на космические лучи, идентифицированные LACosmic, через момент.

Ожидайте, что приведённый ниже код будет выполняться как минимум несколько десятков секунд.

In [ ]:
%%time
new_ccd = ccdp.cosmicray_lacosmic(ccd, readnoise=10, sigclip=7, verbose=True)

Маска `new_ccd` включает как космические лучи, идентифицированные `cosmicray_lacosmic`, так и маску, которую мы применили к `ccd` выше. Чтобы получить только космические лучи, мы устанавливаем маску `False` для всех пикселей, которые были замаскированы перед запуском LACosmic.

In [ ]:
cr_mask = new_ccd.mask
cr_mask[ccd.mask] = False

Сумма маски указывает, сколько пикселей было идентифицировано как космические лучи.

In [ ]:
new_ccd.mask.sum() 

### Исследование космических лучей, идентифицированных LACosmic

Есть 70 пикселей, которые были помечены как космические лучи. Просмотр каждого из них отдельно был бы утомителен в лучшем случае. Было бы также предположительно сложно решить визуально, был ли один пиксель, помеченный как космический луч, на самом деле космическим лучом, но было бы полезно посмотреть на более крупные космические лучи (то есть те, которые охватывают несколько пикселей).

Чтобы идентифицировать эти более крупные космические лучи, мы будем использовать функцию `detect_sources` из пакета [photutils](https://photutils.readthedocs.io), которая идентифицирует смежные пиксели в изображении через сегментацию изображения. Хотя [`detect_sources`](https://photutils.readthedocs.io/en/stable/api/photutils.segmentation.detect_sources.html#photutils.segmentation.detect_sources) предназначена для обнаружения расширенных или звёздных источников в изображении, она оказывается очень хорошо подходящей для идентификации расширенных космических лучей в маске, сгенерированной [`cosmicray_lacosmic`](https://ccdproc.readthedocs.io/en/latest/api/ccdproc.cosmicray_lacosmic.html#ccdproc.cosmicray_lacosmic).

Порог ниже должен быть чем-то меньше 1, чтобы обеспечить, что только замаскированные пиксели (то есть те, чьи значения равны 1) включены как источники. Количество пикселей - это количество, которое должно быть смежным (либо по краю, либо по углу), чтобы считаться источником.

In [ ]:
threshold = 0.5
n_pixels = 3
crs = detect_sources(new_ccd.mask, threshold, n_pixels)

Мы сначала проверяем, сколько пикселей, идентифицированных LACosmic, являются частью этих расширенных космических лучей.

In [ ]:
crs.areas

Похоже, что примерно 50% пикселей, помеченных как космические лучи, расширены на несколько соседних пикселей.

В этом конкретном научном изображении есть три вещи, которые идентифицируются как космические лучи:

+ Фактические космические лучи.
+ Отдельные горячие пиксели (то есть пиксели с необычно высоким тёмным током).

Эти выводы совсем не ясны из того, что мы обсудили в этом ноутбуке до сих пор. Они основаны на детальном исследовании изображений после просмотра миниатюр маски космических лучей, научного изображения, в котором были обнаружены космические лучи, и комбинированного тёмного кадра, который был использован для калибровки этого научного изображения.

Этот вид миниатюр оказался достаточно полезным сравнением, что функция, называемая `display_cosmic_rays`, предоставляется в `convenience_functions.py`, которая отобразит маску космических лучей и столько дополнительных сравниваемых изображений, сколько вам нравится.

Поскольку одно из изображений, которое оказывается полезным для просмотра в этом примере, - это комбинированное тёмное, используемое для калибровки научного изображения, мы его загружаем.

In [ ]:
ccd_dark = CCDData.read(ex2_path / 'combined_dark_90.000.fit')

Одно замечание об аргументе `only_display_rays` ниже, который ограничивает космические лучи, которые отображаются, этим списком. Список был выбран путём сначала просмотра *всех* космических лучей и выбора репрезентативной выборки для включения в это обсуждение. Отобразите их все, установив `only_display_rays=None` в вызове `display_cosmic_rays`.

In [ ]:
images_to_display = [new_ccd.mask, ccd, ccd_dark]
image_titles = ['Mask', 'Science image', 'Combined dark']
display_cosmic_rays(crs, images_to_display, titles=image_titles,
                    only_display_rays=[0, 1, 14, 18]
                   )

### Обсуждение образцов космических лучей

Первые три примера выше, помеченные как "Cosmic ray 0", "Cosmic ray 1" и "Cosmic ray 14", ясны; каждый из них фактически является космическим лучом.

Четвёртый, "Cosmic ray 18", не является космическим лучом, хотя соответствует дефекту CCD. Он вызван одним горячим пикселем (тёмный ток около 2$e^-$/сек), который имеет высокое значение в комбинированном тёмном кадре. Когда этот комбинированный тёмный вычитается из научного изображения, это вызывает большое *отрицательное* значение в значении научного изображения, которое в итоге идентифицируется как космический луч.

### Сохранение маски с изображением

Чтобы сохранить полную маску, включая космические лучи, горячие пиксели и пиксели, идентифицированные `ccdmask`, установите маску `ccd` на маску `new_ccd`. В некоторых случаях использования вы можете предпочесть сохранить сам `new_ccd`. Разница между ними в том, что значения пикселей, в которых есть космические лучи, были заменены в `new_ccd` значениями, репрезентативными для окружающих пикселей.

In [ ]:
ccd.mask = new_ccd.mask

# This saves both the image and the mask
ccd.write('example-with-cosmic-rays.fits')

## Что происходит, если вы не маскируете?

В приведённом выше примере мы замаскировали пиксели, известные как плохие, перед обнаружением космических лучей. Возможно выполнить обнаружение космических лучей без предварительного маскирования. Недостатком отсутствия маскирования является то, что многие характеристики, которые не являются космическими лучами, идентифицируются как космические лучи, и некоторые реальные космические лучи не обнаруживаются.

Мы начинаем со свежей копии входного изображения и запускаем обнаружение космических лучей с теми же параметрами, что и выше.

In [ ]:
ccd = CCDData.read(ex2_path / 'kelt-16-b-S001-R001-C084-r.fit')

In [ ]:
%%time
new_ccd_no_premask = ccdp.cosmicray_lacosmic(ccd, readnoise=10, sigclip=7, verbose=True)

In [ ]:
print("Pre-masking detects {} cosmic ray pixels.\nNo pre-masking detects {} cosmic ray pixels.".format(new_ccd.mask.sum(), new_ccd_no_premask.mask.sum()))

Многие дополнительные пиксели, обнаруженные как космические лучи, находятся в левом столбце изображения. Столбец фактически плохой (он закрыт на CCD и не получает свет).

In [ ]:
crs_no_premask = detect_sources(new_ccd_no_premask.mask, threshold, n_pixels)

In [ ]:
crs_no_premask.areas

### Некоторые из них - не космические лучи

Отображение ниже показывает несколько примеров областей, идентифицированных `cosmicray_lacosmic`, которые фактически не являются космическими лучами. Эти ложные срабатывания безвредны, потому что они отражают реальные проблемы с детектором и всё равно должны быть замаскированы.

Более проблематичны космические лучи, которые не обнаруживаются, если изображение не маскируется в первую очередь. Например, космический луч, помеченный как "Cosmic ray 0" в [примере выше, в котором маска была применена перед обнаружением космических лучей](#discussion-of-sample-cosmic-rays), вообще не обнаруживается при отсутствии маскирования.

In [ ]:
images_to_display = [new_ccd_no_premask.mask, ccd, ccd_dark]
image_titles = ['Mask', 'Science image', 'Combined dark']
display_cosmic_rays(crs_no_premask, images_to_display, titles=image_titles,
                    only_display_rays=[0, 1, 2]
                   )

### Обсуждение

Первый пример выше - это плохой столбец на левой стороне изображения.

Пример с меткой "Cosmic ray 1" вызван одним горячим пикселем с большим отсчётом в тёмных кадрах, что приводит к большому отрицательному значению в откалиброванном научном изображении. LACosmic помечает этот пиксель и многие вокруг него как космические лучи. Хотя отдельный горячий пиксель должен быть замаскирован, окружающие его не нуждаются в маскировании.

Финальный пример, "Cosmic ray 2", выглядит как звезда. На самом деле это остаточное изображение одной из ярких звёзд в поле зрения KELT-16 b. Яркие звёзды в поле зрения могут депонировать достаточно заряда, чтобы он не рассеялся между изображениями. Обычно эффект можно избежать вообще, используя "pre-flashing" CCD.

Поскольку вы не можете выполнить pre-flash после того, как изображения были сняты, мы вынуждены делать следующее лучшее: маскировать эту часть изображений. Маскирование должно быть выполнено на этапе, на котором идентифицируются горячие пиксели, потому что все эти пиксели будут идентифицированы как горячие.